# Aditya-L1 Solar Flare Forecasting & Nowcasting
## CNN-BiLSTM-Attention Model for SoLEXS + HEL1OS Time-Series

**Bharatiya Antariksh Hackathon 2026 — Problem Statement 15**

This notebook demonstrates the end-to-end pipeline for solar flare nowcasting and forecasting using combined soft (SoLEXS) and hard (HEL1OS) X-ray data from ISRO's Aditya-L1 mission.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from config import load_config
from src.data import GOESDataIngester, FlarePreprocessor
from src.data.dataset import create_labels_from_flares, create_data_loaders
from src.models import FlareForecastModel, FocalLoss
from src.nowcasting import Nowcaster, FlareCatalogueMerger
from src.evaluation import FlareEvaluationMetrics
from src.train import Trainer
from src.visualization import LightCurvePlotter, MetricsPlotter
from src.utils import set_seed, setup_logging

config = load_config()
set_seed(42)
setup_logging('INFO')

## 1. Data Ingestion
Loading GOES XRS bootstrap data (2003-2023, 151,071 flares per Hassani et al. 2025)

In [ ]:
goes_ingester = GOESDataIngester(
    config['data']['goes_start_date'],
    config['data']['goes_end_date'],
    config['data']['goes_cadence_seconds']
)
goes_flux = goes_ingester.fetch_goes_xrs()
flare_catalogue = goes_ingester.fetch_hek_flare_catalogue()

print(f"GOES XRS samples: {len(goes_flux)}")
print(f"Flare events: {len(flare_catalogue)}")
df = pd.DataFrame({
    'soft_flux': goes_flux['xrs_b_flux'].values,
    'hard_flux': goes_flux['xrs_a_flux'].values
}, index=goes_flux.index)
df.head()

## 2. Preprocessing & Feature Engineering
Physics-informed features: hardness ratio, Neupert derivatives, rolling statistics

In [ ]:
preproc = FlarePreprocessor(config)
df = preproc.handle_gaps(df)
df = preproc.compute_features(df)
df, params = preproc.standardize(df)

print(f"Features: {list(df.columns)}")
print(f"Shape: {df.shape}")
df.iloc[:5, :8]

## 3. Nowcasting (Threshold-Based Detection)
Real-time detection using 3-sigma threshold + derivative trigger

In [ ]:
nowcaster = Nowcaster(config)
detections = nowcaster.nowcast(df.index, df['soft_flux'].values, df['hard_flux'].values)
merger = FlareCatalogueMerger(config['nowcasting']['merge_window_seconds'])
merged = merger.merge([
    detections[detections['channel'] == 'SoLEXS'],
    detections[detections['channel'] == 'HEL1OS']
], ['SoLEXS', 'HEL1OS'])

print(f"Master nowcast catalogue: {len(merged)} events")
merged.head(10)

## 4. Training the Forecasting Model
CNN-BiLSTM-Attention architecture with Focal Loss

In [ ]:
feature_cols = [c for c in df.columns if c != 'is_valid']
features = df[feature_cols].values.astype(np.float32)
labels = create_labels_from_flares(
    df, flare_catalogue,
    config['data']['forecast_horizons_minutes']
)

train_loader, val_loader, test_loader, pos_weight = create_data_loaders(
    features, df.index, labels,
    config['data']['forecast_horizons_minutes'], config
)

sample_x, sample_y = next(iter(train_loader))
n_features = sample_x.shape[2]
horizons = config['data']['forecast_horizons_minutes']

model = FlareForecastModel(
    n_features=n_features,
    horizons=horizons,
    cnn_config=config['model']['cnn'],
    lstm_config=config['model']['lstm'],
    dense_config=config['model']['dense'],
)

trainer = Trainer(model, config)
history = trainer.fit(train_loader, val_loader, max_epochs=50)

## 5. Evaluation
TSS, AUC, FAR, Lead Time metrics

In [ ]:
metrics = trainer.evaluate(test_loader)
print(f"\nMean TSS across horizons: {metrics['mean_tss_across_horizons']:.4f}")
print(f"Mean AUC across horizons: {metrics['mean_auc_across_horizons']:.4f}")

## 6. Visualize Results

In [ ]:
plotter = LightCurvePlotter()
plotter.plot_light_curves(
    df.index[:1000], df['soft_flux'].values[:1000],
    df['hard_flux'].values[:1000],
    detections=merged,
    save_path='../outputs/figures/light_curves.png'
)

metrics_plotter = MetricsPlotter()
metrics_plotter.plot_metrics_summary(
    metrics,
    save_path='../outputs/figures/metrics_summary.png'
)

print("Figures saved to outputs/figures/")